# 82514 · Sesión S24 — El almacén como problema de decisión: el valor y Q-learning

**Bloque 8** · lunes 14 de diciembre de 2026 · 2 h  ·  IQS Universitat Ramon Llull

**Qué hace este cuaderno.** El almacén de B6 y B7 como un MDP: la celda es el estado, los ocho rumbos son las acciones, el rumbo se tuerce a veces y cada metro cuesta. La iteración de valor da exactamente el frente de onda de S22; con un robot que se tuerce, la política se aparta de las paredes, y con celdas de 50 cm cambia de puerta. Después, Q-learning aprende lo mismo sin conocer el modelo, y el muro de la tabla.

**Se apoya en:** Sutton y Barto (2018); el frente de onda de S22 (cuaderno S22a); documentación de numpy.

**Cómo usarlo en clase.** Sigue el guion de la sesión S24 en las diapositivas del bloque 8. Ejecuta la primera celda una sola vez al empezar; en Colab tarda unos segundos. Las celdas marcadas **Ejercicio** son para que los trabajen los estudiantes: las soluciones están al final del cuaderno.

---

In [ ]:
# Ejecutar una sola vez al abrir el cuaderno. En Colab tarda unos segundos.
import os, sys, importlib, importlib.metadata as md, subprocess
os.environ.setdefault('SDL_VIDEODRIVER', 'dummy')    # pygame (el dibujo del péndulo) sin pantalla y sin avisos

FIJAS = {'gymnasium': '1.4.0', 'scikit-learn': '1.6.1', 'stable-baselines3': '2.9.0'}   # las del bloque 8
COLAB = {'numpy': '2.4.6', 'scipy': '1.16.3', 'matplotlib': '3.10.0', 'torch': '2.11.0'}  # se comprueban, no se instalan
USA = []             # las del bloque que usa este cuaderno
EN_COLAB = 'google.colab' in sys.modules

def version(p):
    try:
        return md.version(p)
    except md.PackageNotFoundError:
        return None

for paquete in USA:
    if paquete == 'stable-baselines3' and version('torch') is None:
        print('AVISO: no hay torch en este entorno: la sección de PPO no se ejecuta (Colab sí lo trae).')
        continue
    if version(paquete) != FIJAS[paquete]:
        print('Instalando', paquete, FIJAS[paquete], '...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', f'{paquete}=={FIJAS[paquete]}'], check=False)
importlib.invalidate_caches()

import math, random, time
import numpy as np, matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
for p in ['numpy', 'scipy', 'matplotlib'] + USA:
    w = (version(p) or 'no instalada').split('+')[0]
    v = {**COLAB, **FIJAS}[p]
    print(f'{p:18s} {w:12s}', '' if w == v else f'(probado con {v})')
print('en Colab:', EN_COLAB)
if not np.__version__.startswith('2.4'):
    print('AVISO: este cuaderno se probó con numpy 2.4; las cifras con semilla pueden cambiar un poco.')

plt.rcParams['figure.figsize'] = (11, 3.6)
IQS_AZUL, IQS_VERDE, GRIS, AMBAR, ROJO = '#1B2A80', '#1FA355', '#8A8FA8', '#B9770E', '#C0392B'
COMA = FuncFormatter(lambda v, p: f'{v:g}'.replace('.', ',').replace('-', '−'))   # ejes con coma decimal

def coma(x, n=2):
    """Un número con coma decimal y signo menos, para imprimir."""
    return f'{x:.{n}f}'.replace('.', ',').replace('-', '−')

def pc(x):
    """Una fracción como porcentaje entero."""
    return f'{100 * x:.0f} %'

> **Aviso para Colab.** Ábrelo al empezar la sesión y ejecuta la primera celda: no instala nada (usa numpy, scipy y matplotlib de Colab) y tarda unos segundos.
>
> **En clase (diez minutos):** la sección 1 sin detenerte, y el taller, de la 2 a la 5. Las cifras tienen que salir como en las diapositivas.
>
> **En casa:** la sección 6 (el muro de la tabla) y los ejercicios 1 a 3; las soluciones están al final.
>
> **No se sube:** este cuaderno no tiene tarea en Moodle.

## 1. El almacén como MDP (diapositiva 7)

*En clase, ejecuta esta celda sin detenerte.*

Es el almacén de B6 y B7: 13 × 9 m, el muelle (0,6; 0,6), el picking (8,3; 8,2) y el puesto de la derecha (11,5; 6,0), la puerta ancha y la estrecha. Como en S22, el robot, de 0,28 m de radio, es un punto y los obstáculos van inflados con ese radio.

- **Estado s:** la celda en la que está el robot. Se trabaja a tres resoluciones: 10 cm (la rejilla de S22, 8100 celdas libres), 50 cm (333) y 20 cm (1974).
- **Acción a:** uno de los ocho rumbos (E, NE, N, NO, O, SO, S y SE).
- **Transición p(s′ | s, a):** el rumbo pedido sale con 1 − 2·DESVIO y se tuerce 45° con DESVIO a cada lado. Si el paso lleva a una celda no libre, el robot se queda donde está.
- **Recompensa r:** menos los metros del paso (0,1 o 0,141 m a 10 cm); si choca, menos CHOQUE metros.
- **Descuento:** γ = 1, porque el viaje acaba en la meta. Por eso **−V es el coste hasta la meta, en metros**.

La celda pega el código del almacén de S22 y el del MDP: el mismo que usan las diapositivas.

In [ ]:
# El almacén de S22 y el MDP (el mismo código que las diapositivas). Sin detenerse.
import math, heapq
import numpy as np
from scipy.ndimage import distance_transform_edt, binary_erosion

RES, FILAS, COLS = 0.10, 90, 130             # celdas de 10 cm: 13 m en x, 9 m en y


def mapa_almacen():
    """La rejilla de ocupacion del almacen de B6 (True = ocupado). Fila = y/0,1; columna = x/0,1: la celda i esta
    centrada en i·0,1 m (es un cuadrado de 10 cm, de i·0,1 − 0,05 a i·0,1 + 0,05)."""
    occ = np.zeros((FILAS, COLS), bool)
    occ[0, :] = occ[-1, :] = True              # paredes exteriores
    occ[:, 0] = occ[:, -1] = True
    occ[42:47, :] = True                       # el muro central, de y = 4,2 a 4,7 m
    occ[42:47, 8:28] = False                   #   la puerta ancha, de x = 0,8 a 2,8 m
    occ[42:47, 92:100] = False                 #   la puerta estrecha, de x = 9,2 a 10,0 m
    occ[12:36, 40:47] = True                   # estanterias de la nave de abajo
    occ[12:36, 64:71] = True
    occ[58:78, 34:41] = True                   # estanterias de la nave de arriba
    occ[58:78, 58:65] = True
    occ[58:78, 82:89] = True
    return occ


def con_estrecha_cerrada(occ):
    """El mismo almacen con la puerta estrecha cerrada (un palé delante, como en S18)."""
    o = occ.copy()
    o[42:47, 92:100] = True
    return o


OCC = mapa_almacen()
DIST = distance_transform_edt(~OCC) * RES    # distancia de centro a centro de cada celda a la ocupada mas cercana, en m
EXT = (-RES / 2, COLS * RES - RES / 2, -RES / 2, FILAS * RES - RES / 2)   # para imshow: la celda i, centrada en i·0,1 m
MUELLE, PICKING, DERECHA = (0.6, 0.6), (8.3, 8.2), (11.5, 6.0)   # los puntos con nombre del almacen, en m
R_ROBOT, V_MAX, W_MAX = 0.28, 0.5, 1.0       # el robot del almacen: radio, 0,5 m/s y 1 rad/s


def celda(p):
    """De (x; y) en metros a (fila, columna) de la rejilla."""
    return (int(round(p[1] / RES)), int(round(p[0] / RES)))


def libre_para(occ, r):
    """El espacio libre para el CENTRO de un robot redondo de radio r: los obstaculos, inflados con r."""
    return distance_transform_edt(~occ) * RES > r


VECINOS = [(-1, 0, 1.0), (1, 0, 1.0), (0, -1, 1.0), (0, 1, 1.0),
           (-1, -1, math.sqrt(2)), (-1, 1, math.sqrt(2)), (1, -1, math.sqrt(2)), (1, 1, math.sqrt(2))]


def buscar(coste, ini, meta, peso=1.0, devolver_cerrada=False):
    """A* sobre la rejilla, con 8 vecinos. coste[v] es lo que cuesta entrar en v (inf = no se puede).
    peso = 0: Dijkstra (h = 0); 1: A* con la euclidea, admisible; 2: la heuristica doblada, que se pasa.
    Devuelve (camino, coste, nodos expandidos[, conjunto de expandidos])."""
    h = lambda v: peso * math.hypot(v[0] - meta[0], v[1] - meta[1])
    n, m = coste.shape
    g, padre = {ini: 0.0}, {}
    abierta, cerrada = [(h(ini), 0.0, ini)], set()   # OPEN, ordenada por f = g + h; CLOSED
    while abierta:
        _, gu, u = heapq.heappop(abierta)
        if u in cerrada:
            continue
        cerrada.add(u)
        if u == meta:
            break
        for di, dj, paso in VECINOS:
            v = (u[0] + di, u[1] + dj)
            if not (0 <= v[0] < n and 0 <= v[1] < m) or v in cerrada:
                continue
            c = coste[v]
            if not np.isfinite(c):
                continue
            ng = gu + paso * c
            if ng < g.get(v, np.inf):
                g[v] = ng
                padre[v] = u
                heapq.heappush(abierta, (ng + h(v), ng, v))
    if meta not in g:
        res = (None, np.inf, len(cerrada))
    else:
        cam, u = [meta], meta
        while u != ini:
            u = padre[u]
            cam.append(u)
        res = (cam[::-1], g[meta], len(cerrada))
    return res + (cerrada,) if devolver_cerrada else res


def frente_de_onda(coste, meta):
    """Dijkstra desde la meta hacia todas partes: el coste de cada celda hasta la meta (la transformada de distancia)."""
    D = np.full(coste.shape, np.inf)
    D[meta] = 0.0
    monton = [(0.0, meta)]
    while monton:
        d, u = heapq.heappop(monton)
        if d > D[u]:
            continue
        for di, dj, paso in VECINOS:
            v = (u[0] + di, u[1] + dj)
            if not (0 <= v[0] < coste.shape[0] and 0 <= v[1] < coste.shape[1]):
                continue
            if not np.isfinite(coste[v]):
                continue
            nd = d + paso * coste[v]
            if nd < D[v]:
                D[v] = nd
                heapq.heappush(monton, (nd, v))
    return D


def bajar(D, ini, max_pasos=20000):
    """El camino desde cualquier celda: bajar por el frente de onda, al vecino de menor D, hasta la meta (D = 0)."""
    cam, u = [ini], ini
    for _ in range(max_pasos):
        if D[u] == 0.0:
            return cam
        v = min(((u[0] + di, u[1] + dj) for di, dj, _ in VECINOS
                 if 0 <= u[0] + di < D.shape[0] and 0 <= u[1] + dj < D.shape[1]), key=lambda w: D[w])
        if D[v] >= D[u]:
            return None
        u = v
        cam.append(u)
    return None


def largo(cam):
    """Longitud de un camino de celdas, en metros."""
    p = np.array(cam, float) * RES
    return float(np.linalg.norm(np.diff(p, axis=0), axis=1).sum())


_CARAS = np.argwhere(OCC & ~binary_erosion(OCC, border_value=0)) * RES   # (y, x) de las celdas ocupadas del borde


def a_la_pared(xy, caras=None):
    """La distancia (m) de cada punto (x, y) a la cara del obstaculo mas cercano: cada celda ocupada es un cuadrado
    de 10 cm, y se mide hasta su borde, no hasta su centro (que esta 5 cm mas alla)."""
    p = np.atleast_2d(np.asarray(xy, float))
    caras = _CARAS if caras is None else caras
    out = np.empty(len(p))
    for k in range(0, len(p), 500):                              # por tandas, para no gastar memoria
        q = p[k:k + 500]
        dx = np.maximum(np.abs(q[:, None, 0] - caras[None, :, 1]) - RES / 2, 0.0)
        dy = np.maximum(np.abs(q[:, None, 1] - caras[None, :, 0]) - RES / 2, 0.0)
        out[k:k + 500] = np.hypot(dx, dy).min(axis=1)
    return out


def holgura(cam):
    """La distancia minima del centro del robot a la pared (a la cara del obstaculo) a lo largo del camino, en m."""
    c = np.array(cam, float)
    return float(a_la_pared(c[:, ::-1] * RES).min())


def puerta(cam):
    """Por que puerta cruza el muro un camino de celdas."""
    cols = [cc for f, cc in cam if 42 <= f <= 46]
    return "ancha" if cols and max(cols) < 40 else ("estrecha" if cols else "ninguna")


import math, random
import numpy as np

# Los ocho rumbos, en (fila, columna) de la rejilla (la fila es y): E, NE, N, NO, O, SO, S y SE. Sumar 1 al índice
# es girar 45° a la izquierda; restar 1, a la derecha.
RUMBOS = [(0, 1), (1, 1), (1, 0), (1, -1), (0, -1), (-1, -1), (-1, 0), (-1, 1)]
NOMBRES = ['E', 'NE', 'N', 'NO', 'O', 'SO', 'S', 'SE']
LIBRE = libre_para(OCC, R_ROBOT)       # dónde puede estar el centro del robot (S22): 8100 celdas de 10 cm
_L = np.argwhere(LIBRE)
A_PARED = np.full(OCC.shape, np.nan)   # la distancia de cada celda libre a la cara de la pared, como en S22
A_PARED[LIBRE] = a_la_pared(_L[:, ::-1] * RES)


class Rejilla:
    """El almacén con celdas de k·10 cm. La celda (i, j) es la del centro (k·i, k·j) de la rejilla fina de S22, y es
    libre si ese centro está en el espacio libre del robot (los obstáculos, inflados con 0,28 m).
    Para cada estado s y rumbo a: nxt[s, a] es el estado al que se llega (el mismo s si el paso choca), ok[s, a] dice
    si el paso es posible y lng[s, a] es su largo en metros (k·0,1 recto, k·0,1·√2 en diagonal)."""

    def __init__(self, k):
        self.k, self.res = k, k * RES
        self.libre = LIBRE[::k, ::k].copy()
        self.celdas = np.argwhere(self.libre)
        self.n = len(self.celdas)
        self.idx = -np.ones(self.libre.shape, int)
        self.idx[self.libre] = np.arange(self.n)
        F, C = self.libre.shape
        self.nxt = np.empty((self.n, 8), int)
        self.ok = np.empty((self.n, 8), bool)
        self.lng = np.empty((self.n, 8))
        for a, (di, dj) in enumerate(RUMBOS):
            f, c = self.celdas[:, 0] + di, self.celdas[:, 1] + dj
            dentro = (f >= 0) & (f < F) & (c >= 0) & (c < C)
            fr, cr = np.where(dentro, f, 0), np.where(dentro, c, 0)
            libre = dentro & self.libre[fr, cr]
            self.nxt[:, a] = np.where(libre, self.idx[fr, cr], np.arange(self.n))
            self.ok[:, a] = libre
            self.lng[:, a] = self.res * math.hypot(di, dj)
        self.pared = A_PARED[self.celdas[:, 0] * k, self.celdas[:, 1] * k]   # a la cara de la pared, en m

    def estado(self, p):
        """El estado (la celda) de un punto (x; y) en metros."""
        return self.idx[int(round(p[1] / self.res)), int(round(p[0] / self.res))]

    def xy(self, s):
        """El centro de la celda s, en metros."""
        return round(float(self.celdas[s, 1] * self.res), 3), round(float(self.celdas[s, 0] * self.res), 3)


def recompensas(R, choque=1.0):
    """r(s, a): menos los metros del paso; si el paso choca, el robot se queda donde está y paga 'choque' metros."""
    return np.where(R.ok, -R.lng, -choque)


def iteracion_valor(R, meta, desvio=0.0, choque=1.0, V0=None, tol=1e-10, maxit=100000, termina=False, fotos=()):
    """La iteración de valor con γ = 1: V(s) ← máx_a Σ p(s′|s, a)·[r + V(s′)], con V(meta) = 0.
    La acción sale bien con 1 − 2·desvio y se tuerce 45° a cada lado con 'desvio'. Al principio solo la meta tiene
    valor (las demás, −∞). termina=True: un choque acaba el viaje (y paga 'choque'). Devuelve V, Q (n × 8), los
    barridos hasta que no cambia nada y, si se piden, las fotos {barrido: V}."""
    r = recompensas(R, choque)
    V = np.full(R.n, -np.inf) if V0 is None else V0.copy()
    V[meta] = 0.0
    sacadas = {}
    for k in range(1, maxit + 1):
        Vs = V[R.nxt]
        if termina:
            Vs = np.where(R.ok, Vs, 0.0)
        T = r + Vs                                          # lo que vale cada rumbo si sale tal cual
        if desvio == 0:
            Q = T
        else:
            Q = (1 - 2 * desvio) * T + desvio * np.roll(T, -1, axis=1) + desvio * np.roll(T, 1, axis=1)
        Q[meta] = 0.0
        Vn = Q.max(axis=1)
        if k in fotos:
            sacadas[k] = Vn.copy()
        fa, fb = np.isfinite(V), np.isfinite(Vn)
        if np.array_equal(fa, fb) and (not fb.any() or np.max(np.abs(Vn[fb] - V[fb])) < tol):
            return Vn, Q, k, sacadas
        V = Vn
    return V, Q, maxit, sacadas


def recorrer(R, pol, ini, meta, desvio, rng, nmax=3000):
    """Un viaje con deslizamiento siguiendo la política pol (un rumbo por estado). Devuelve (llega, metros
    recorridos, choques, distancia mínima a la pared y las celdas visitadas)."""
    s, L, ch, cam = ini, 0.0, 0, [ini]
    for _ in range(nmax):
        if s == meta:
            break
        a = pol[s]
        u = rng.random()
        d = a if u < 1 - 2 * desvio else ((a + 1) % 8 if u < 1 - desvio else (a - 1) % 8)
        if R.ok[s, d]:
            L += R.lng[s, d]
            s = R.nxt[s, d]
            cam.append(s)
        else:
            ch += 1
    return s == meta, L, ch, float(R.pared[cam].min()), cam


def puerta_de(R, cam):
    """Por qué puerta cruza el muro un camino de estados."""
    filas, cols = R.celdas[cam, 0] * R.k, R.celdas[cam, 1] * R.k
    m = (filas >= 42) & (filas <= 46)
    if not m.any():
        return 'ninguna'
    return 'ancha' if cols[m].max() < 50 else 'estrecha'


def viajes(R, pol, ini, meta, desvio, n=500, semilla=1):
    """n viajes con deslizamiento: (fracción que llega, metros, choques por viaje, fracción de viajes limpios,
    mediana de la distancia mínima a la pared, fracción por la puerta estrecha)."""
    rng = random.Random(semilla)
    res = [recorrer(R, pol, ini, meta, desvio, rng) for _ in range(n)]
    return (np.mean([x[0] for x in res]), np.mean([x[1] for x in res]), np.mean([x[2] for x in res]),
            np.mean([x[2] == 0 for x in res]), np.median([x[3] for x in res]),
            np.mean([puerta_de(R, x[4]) == 'estrecha' for x in res]))


def camino_nominal(R, pol, ini, meta):
    """La política seguida sin deslizamiento: el camino que «planea» y su largo en metros."""
    s, cam, L = ini, [ini], 0.0
    for _ in range(5000):
        if s == meta:
            break
        a = pol[s]
        if not R.ok[s, a]:
            return None, np.inf
        L += R.lng[s, a]
        s = R.nxt[s, a]
        cam.append(s)
    return cam, L


def q_learning(R, meta, desvio, choque, pasos, alfa=0.2, eps0=0.5, eps1=0.05, semilla=7, tope=300):
    """Q-learning sin el modelo: el robot solo vive transiciones (s, a, r, s′). Sale de una celda al azar, elige con
    ε-voraz (ε baja de eps0 a eps1, lineal) y actualiza Q(s,a) ← Q(s,a) + α·[r + γ·máx Q(s′,a′) − Q(s,a)], con γ = 1 y
    máx Q(s′,a′) = 0 en la meta. Cada episodio acaba en la meta o a los 'tope' pasos. Devuelve Q, episodios y segundos."""
    import time
    rng = random.Random(semilla)
    Q = [[0.0] * 8 for _ in range(R.n)]
    nxt = R.nxt.tolist()
    r = recompensas(R, choque).tolist()
    k = ep = 0
    t0 = time.perf_counter()
    while k < pasos:
        s = rng.randrange(R.n)
        ep += 1
        for _ in range(tope):
            eps = eps0 + (eps1 - eps0) * k / pasos
            q = Q[s]
            a = rng.randrange(8) if rng.random() < eps else max(range(8), key=q.__getitem__)   # ε-voraz
            u = rng.random()                                                                    # el deslizamiento
            d = a if u < 1 - 2 * desvio else ((a + 1) % 8 if u < 1 - desvio else (a - 1) % 8)
            s2 = nxt[s][d]
            objetivo = r[s][d] if s2 == meta else r[s][d] + max(Q[s2])
            q[a] += alfa * (objetivo - q[a])                                                    # la sorpresa, por α
            s = s2
            k += 1
            if s == meta:
                break
    return np.array(Q), ep, time.perf_counter() - t0


def comparar(R, meta, Qa, desvio, choque, V0):
    """Lo aprendido frente a la iteración de valor: fracción de estados con un rumbo casi óptimo (a 5 cm del mejor),
    con el mismo rumbo, el error medio |máx Q − V| en metros, y V."""
    V, Qv, _, _ = iteracion_valor(R, meta, desvio, choque, V0=V0)
    nt = np.array([s for s in range(R.n) if s != meta])
    a = Qa.argmax(1)
    casi = (Qv[nt, a[nt]] >= V[nt] - 0.05).mean()
    misma = (a[nt] == Qv.argmax(1)[nt]).mean()
    return casi, misma, np.abs(Qa.max(1)[nt] - V[nt]).mean(), V


R10, R20, R50 = Rejilla(1), Rejilla(2), Rejilla(5)
for R in (R10, R50, R20):
    print(f'celdas de {R.res * 100:.0f} cm: {R.n} estados × 8 rumbos = {8 * R.n} pares (s, a)')


def camino_xy(R, cam):
    c = R.celdas[cam]
    return c[:, 1] * R.res, c[:, 0] * R.res


def dibuja_almacen(ax, titulo='', lim=EXT):
    img = np.ones(OCC.shape + (3,))
    img[~LIBRE & ~OCC] = (0.96, 0.84, 0.82)
    img[OCC] = (0.29, 0.31, 0.39)
    ax.imshow(img, origin='lower', extent=EXT, interpolation='nearest')
    ax.set_xlim(lim[0], lim[1]); ax.set_ylim(lim[2], lim[3]); ax.set_aspect('equal')
    ax.xaxis.set_major_formatter(COMA); ax.yaxis.set_major_formatter(COMA)
    ax.set_title(titulo, fontsize=10)


def coste(ax, R, V):
    M = np.full(R.libre.shape, np.nan)
    M[R.celdas[:, 0], R.celdas[:, 1]] = -V
    ext = (-R.res / 2, R.libre.shape[1] * R.res - R.res / 2, -R.res / 2, R.libre.shape[0] * R.res - R.res / 2)
    return ax.imshow(np.ma.masked_invalid(M), origin='lower', extent=ext, cmap='Blues', interpolation='nearest')


def flechas(ax, R, Q, color=IQS_AZUL):
    d = np.array(RUMBOS)[Q.argmax(1)]
    n = np.hypot(d[:, 0], d[:, 1])
    ax.quiver(R.celdas[:, 1] * R.res, R.celdas[:, 0] * R.res, d[:, 1] / n, d[:, 0] / n, color=color,
              scale=1 / (0.36 * R.res), scale_units='xy', angles='xy', width=0.004, pivot='middle')


muelle, picking, derecha = R10.estado(MUELLE), R10.estado(PICKING), R10.estado(DERECHA)

## 2. Taller: la iteración de valor es el frente de onda de S22 (diapositivas 10 y 11)

La iteración de valor aplica la ecuación de Bellman a todas las celdas, una y otra vez:

V(s) ← máx_a Σ p(s′ | s, a)·[r + γ·V(s′)]

Al principio solo la meta tiene valor (las demás, −∞). Sin desvío y con γ = 1 es la relajación de Dijkstra de S22, hecha en todas las celdas a la vez en cada barrido: el algoritmo de Bellman-Ford. Tras k barridos tienen valor las celdas que están a k pasos o menos de la meta.

**Taller.** Ejecuta: tiene que salir 12,664 m desde el muelle, la cifra del frente de onda de S22, igual en las 8100 celdas.

In [ ]:
V, Q, k, fotos = iteracion_valor(R10, picking, fotos=(10, 50, 100, 112))
D = frente_de_onda(np.where(LIBRE, 1.0, np.inf), celda(PICKING)) * RES      # el frente de onda de S22
Dv = D[R10.celdas[:, 0], R10.celdas[:, 1]]
print(f'-V(muelle) = {coma(-V[muelle], 3)} m en {k} barridos')
print(f'el frente de onda de S22: {coma(Dv[muelle], 3)} m')
print(f'diferencia máxima: {np.max(np.abs(Dv + V)):.0e} m'.replace('.', ','))

fig, axs = plt.subplots(1, 4, figsize=(13, 2.8))
for ax, kk in zip(axs, (10, 50, 100, 112)):
    dibuja_almacen(ax, f'{kk} barridos: {int(np.isfinite(fotos[kk]).sum())} celdas con valor')
    coste(ax, R10, fotos[kk])
plt.tight_layout(); plt.show()

## 3. Taller: con desvío, la política se aparta de las paredes (diapositiva 12)

Ahora el rumbo se tuerce 45° con DESVIO a cada lado, y chocar cuesta CHOQUE metros (el robot se queda donde está). La iteración de valor pondera las tres salidas:

Q(s, a) = (1 − 2·DESVIO)·[r + V(s_a)] + DESVIO·[r + V(s_a+45°)] + DESVIO·[r + V(s_a−45°)]

Se compara con la política del frente de onda, que no sabe que el robot se tuerce, en 500 viajes desde el muelle hasta el puesto de la derecha. La distancia a la pared se mide a la cara del obstáculo, como en S22. Recuerda el radio del robot, 0,28 m: por debajo, el disco del robot pisa la pared (la media celda de S22).

**Taller.** Ejecuta con DESVIO = 0,10 y CHOQUE = 1,0. ¿Cuántos choques? ¿Por qué puerta? ¿A cuánto de la pared? Si te sobra tiempo, prueba 0,05 y 0,5.

In [ ]:
DESVIO, CHOQUE = 0.10, 1.0      # prueba 0.05 y 0.5

Vd, Qd, _, _ = iteracion_valor(R10, derecha)                                   # el frente de onda, sin desvío
Vs, Qs, k, _ = iteracion_valor(R10, derecha, DESVIO, CHOQUE, V0=Vd)              # sabe que se tuerce
fo = viajes(R10, Qd.argmax(1), muelle, derecha, DESVIO)
vi = viajes(R10, Qs.argmax(1), muelle, derecha, DESVIO)
c_fo, L_fo = camino_nominal(R10, Qd.argmax(1), muelle, derecha)
c_vi, L_vi = camino_nominal(R10, Qs.argmax(1), muelle, derecha)
print(f'frente de onda: {coma(fo[2])} choques por viaje')
print(f'con desvío: {coma(vi[2])} choques; {coma(L_vi)} m')
print(f'a la pared: de {coma(R10.pared[c_fo].min())} a {coma(R10.pared[c_vi].min())} m')
print(f'(viajes limpios: {pc(fo[3])} y {pc(vi[3])}; puerta: {puerta_de(R10, c_fo)} y {puerta_de(R10, c_vi)}; {k} barridos; '
      f'en los viajes con desvío, la que se aparta pasa a {coma(vi[4])} m de la pared, en la mediana)')

fig, ax = plt.subplots(figsize=(5, 4))
dibuja_almacen(ax, 'La puerta estrecha', lim=(8.4, 11.2, 3.5, 5.9))
ax.plot(*camino_xy(R10, c_fo), '--', color=ROJO, lw=2, label='frente de onda')
ax.plot(*camino_xy(R10, c_vi), color=AMBAR, lw=2.5, label='con desvío')
ax.legend(fontsize=8); plt.show()

Ojo con los números. El frente de onda pasa a 0,21 m de la cara de la pared: menos que el radio, porque la rejilla de S22 es optimista en media celda. Y «choque», en el MDP, es pisar una celda que no es libre, no tocar la pared: la política que se aparta planea a 0,29 m, pero en los viajes con desvío pasa, en la mediana, a 0,25 m (la última línea). Un robot real rozaría la pared en más de la mitad de los viajes: la mediana ya está por debajo del radio. Para tener margen de verdad, se infla con el radio más media celda: 0,33 m.

## 4. Taller: a 50 cm, cambia de puerta (diapositivas 10 y 12)

Con celdas de 50 cm, un paso torcido se aparta medio metro, más que el hueco de la puerta estrecha. Las flechas son la política: un rumbo por celda; el color, el coste hasta la meta, −V.

Con desvío 0,10 y choque de 1 m, los de la diapositiva 12, aunque en la sección 3 hayas probado otros.

**Taller.** Ejecuta. ¿Por qué puerta va cada una? ¿Cuántas flechas cambian?

In [ ]:
m50, d50 = R50.estado(MUELLE), R50.estado(DERECHA)
V50, Q50, _, _ = iteracion_valor(R50, d50)
V5s, Q5s, _, _ = iteracion_valor(R50, d50, 0.10, 1.0, V0=V50)     # siempre 0,10 y 1 m, aunque en la §3 hayas probado otros
c1, L1 = camino_nominal(R50, Q50.argmax(1), m50, d50)
c2, L2 = camino_nominal(R50, Q5s.argmax(1), m50, d50)
print(f'frente de onda: {coma(L1)} m por la {puerta_de(R50, c1)}')
print(f'con desvío: {coma(L2)} m por la {puerta_de(R50, c2)}')
print(f'(flechas que cambian: {(Q50.argmax(1) != Q5s.argmax(1)).sum()} de {R50.n})')

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
for ax, (V_, Q_, c_, t_) in zip(axs, ((V50, Q50, c1, 'Sin desvío'), (V5s, Q5s, c2, 'Con desvío'))):
    dibuja_almacen(ax, t_); coste(ax, R50, V_); flechas(ax, R50, Q_)
    ax.plot(*camino_xy(R50, c_), color=AMBAR, lw=2.5)
plt.tight_layout(); plt.show()

## 5. Taller: Q-learning, el valor sin el modelo (diapositivas 13 a 17)

Q-learning no conoce la transición: el robot sale de una celda al azar, elige un rumbo con ε-voraz (ε baja de 0,5 a 0,05), vive la transición (s, a, r, s′) y corrige

Q(s, a) ← Q(s, a) + α·[r + γ·máx Q(s′, a′) − Q(s, a)]

con α = 0,2 y γ = 1. El corchete es la sorpresa (el error de diferencia temporal, TD). Después se compara con la iteración de valor con desvío: un rumbo es «casi óptimo» si su valor está a 5 cm o menos del mejor.

Con desvío 0,10 y choque de 1 m, los de la diapositiva 12.

**Taller.** Ejecuta con PASOS = 600 000 (en un par de segundos). Si te sobra tiempo, prueba 50 000.

In [ ]:
PASOS = 600_000      # prueba 50_000 y 200_000

Qa, ep, t = q_learning(R50, d50, 0.10, 1.0, PASOS)             # 0,10 y 1 m, los de la diapositiva 12
casi, misma, err, Vq = comparar(R50, d50, Qa, 0.10, 1.0, V50)
ll, L, ch, lim, hol, est = viajes(R50, Qa.argmax(1), m50, d50, 0.10, n=300)
print(f'casi óptima en el {pc(casi)}; ' + (f'llega por la {"ancha" if est < 0.5 else "estrecha"}' if ll == 1 else f'llega en el {pc(ll)}'))
viaje = (f'desde el muelle {coma(L)} m y {coma(ch)} choques: un coste de {coma(L + ch)} m, frente a los '
         f'{coma(-Vq[m50])} que espera la iteración de valor' if ll == 1 else 'desde el muelle, no siempre llega')
print(f'({ep} episodios en {coma(t, 1)} s; la misma acción en el {pc(misma)}; {viaje})')

fig, ax = plt.subplots(figsize=(6.5, 4.5))
dibuja_almacen(ax, f'Q-learning, {PASOS} pasos'); flechas(ax, R50, Qa)
c3, _ = camino_nominal(R50, Qa.argmax(1), m50, d50)
if c3:
    ax.plot(*camino_xy(R50, c3), color=AMBAR, lw=2.5)
plt.show()

## 6. En casa: el muro de la tabla (diapositiva 18)

La tabla tiene una entrada por par (s, a): 2664 a 50 cm, 15 792 a 20 cm y 64 800 a 10 cm. Cada celda se aprende por separado, y lo que no se visita no se sabe. Aquí, el mismo Q-learning con celdas de 20 cm (tarda unos segundos).

In [ ]:
m20, d20 = R20.estado(MUELLE), R20.estado(DERECHA)
V20, _, _, _ = iteracion_valor(R20, d20)
for pasos in (200_000, 1_000_000, 3_000_000):
    Qb, ep, t = q_learning(R20, d20, 0.10, 1.0, pasos)
    casi, misma, err, Vb = comparar(R20, d20, Qb, 0.10, 1.0, V20)
    ll = viajes(R20, Qb.argmax(1), m20, d20, 0.10, n=200)[0]
    print(f'{pasos:9d} pasos ({coma(t, 1)} s): casi óptima en el {pc(casi)} de {R20.n} estados; desde el muelle llega el {pc(ll)}')
print(f'un brazo de seis ejes a 2° por eje: 180**6 = {180 ** 6:.1e} celdas, unos {180 ** 6 * 8 / 1e12:.0f} TB'.replace('.', ','))

### Ejercicio 1 (en casa): ¿desde qué castigo deja la puerta estrecha?

Con celdas de 50 cm y la meta a la derecha, recorre CHOQUE de 0 a 1 m (para DESVIO = 0,05 y 0,10) y anota por qué puerta va el camino que planea la política. ¿Desde qué castigo deja la estrecha? ¿Por qué depende del desvío?

In [ ]:
# Ejercicio 1: completa el bucle
for desvio in (0.05, 0.10):
    for choque in (0.25, 0.5, 0.6, 0.7, 1.0):
        pass   # iteracion_valor(R50, d50, desvio, choque, V0=V50), camino_nominal(...) y puerta_de(...)

### Ejercicio 2 (en casa): una recompensa mal puesta

Con celdas de 10 cm, meta en el picking y sin desvío, haz que chocar **acabe el viaje** con un castigo C (`termina=True`). ¿Qué fracción de las celdas prefiere estrellarse con C = 5, 10, 12 y 13 m? ¿Por qué deja de pasar desde unos 13 m?

In [ ]:
# Ejercicio 2: compara V con choque terminal frente al V del frente de onda (el del picking)
Vp, _, _, _ = iteracion_valor(R10, picking)
for C_ in (5.0, 10.0, 12.0, 13.0):
    pass   # Vt, _, _, _ = iteracion_valor(R10, picking, 0.0, C_, termina=True); fracción de celdas con Vt > Vp

### Ejercicio 3 (en casa): explorar poco o mucho

Repite Q-learning con ε fijo (`eps0 = eps1`) de 0,02 y de 0,9, con 600 000 pasos. Compara la fracción de rumbos casi óptimos, el coste del viaje desde el muelle y cuántos episodios completa mientras aprende (cada uno acaba en la meta o a los 300 pasos).

In [ ]:
# Ejercicio 3
for e in (0.02, 0.9):
    pass   # Qe, ep, t = q_learning(R50, d50, 0.10, 1.0, 600_000, eps0=e, eps1=e); comparar(...)

---

## Soluciones

**Ejercicio 1.** Con DESVIO = 0,10, la política planea por la estrecha con un castigo de 0,25 m y por la ancha desde 0,3 m (el cambio está entre 0,25 y 0,3 m; con 1 m, 14,45 m por la ancha y 94 flechas distintas de las del frente de onda). Con DESVIO = 0,05, sigue por la estrecha con 0,5 m y se va a la ancha desde 0,55 m (el cambio está entre 0,5 y 0,55 m). El cambio tiene umbral, y el umbral baja cuando el robot se tuerce más: cuanto más probable es chocar en el hueco, menos castigo hace falta para que compense el rodeo. Es la lógica del inflado de Nav2 en S23 (0,70 m por la ancha; 0,35 m por la estrecha), sacada de la incertidumbre y del precio del choque.

**Ejercicio 2.** Con 5 m, el 66,3 % de las celdas prefiere estrellarse; con 10 m, el 11,7 % (desde el muelle: avanza 0,30 m y choca, 10,30 m frente a 12,66 m llegando); con 12 m, el 1,9 %; con 13 m, casi ninguna: 3 de las 8100 (el 0,04 %), en el rincón junto al muelle, donde el viaje cuesta más de 13 m; desde 13,09 m, ninguna, porque es lo que cuesta el viaje más largo hasta el picking. El agente optimiza lo que se le mide: si chocar acaba el viaje y cuesta menos que llegar, choca. Con el choque del curso (el robot se queda y paga 1 m), estrellarse nunca compensa.

**Ejercicio 3.** Con ε = 0,02, el 87 % de los rumbos casi óptimos y, desde el muelle, 16,98 m y 0,36 choques: un coste de 17,34 m; completa 29 024 episodios, porque casi siempre va directo a la meta. Con ε = 0,9, el 92 %, y desde el muelle 15,80 m y 0,98 choques: 16,77 m, porque explora todo el mapa; pero mientras aprende va casi al azar y solo completa 3106 episodios en los mismos pasos. En un robot real, cada uno de esos pasos al azar es un choque posible: por eso se aprende en simulación.

---

## Para llevarse de esta sesión

**El valor es el coste hasta la meta.** Con γ = 1 y sin desvío, la iteración de valor es el frente de onda de S22: 12,664 m desde el muelle, igual en las 8100 celdas.

**Con incertidumbre, la política se aparta de las paredes.** A 10 cm pasa de 0,21 a 0,29 m de la pared y deja de chocar; a 50 cm cambia de puerta, como Nav2 con el inflado.

**Q-learning aprende lo mismo sin el modelo, a cambio de experiencia,** y la tabla no escala: a 20 cm, con tres millones de pasos, aún no llega.

*Cuaderno del curso 82514 Mecatrónica y Robótica · IQS Universitat Ramon Llull · curso 2026/27*

*© Guillermo Reyes Carmenaty · Publicado bajo [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/deed.es): puedes usarlo, adaptarlo y redistribuirlo, incluso con fines comerciales, siempre que reconozcas la autoría.*